# 1. SupportSense — Exploratory Analysis
## Business context
Northstar Cloud is a fictional B2B SaaS company. Its support team answers recurring questions about account access, billing, security, and product workflows. This notebook inspects the sample approved knowledge base and evaluation set before building a retrieval-augmented assistant.

**Goal:** understand document coverage, test-question coverage, and the limitations of the demonstration data.

> The included company documents and questions are synthetic examples created for this portfolio project; they are not real company or customer records.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
docs = json.loads((ROOT / "data" / "company_docs.json").read_text(encoding="utf-8"))
questions = json.loads((ROOT / "data" / "evaluation_questions.json").read_text(encoding="utf-8"))
docs_df = pd.DataFrame(docs)
questions_df = pd.DataFrame(questions)
print(f"Documents: {len(docs_df)} | Evaluation questions: {len(questions_df)}")
display(docs_df[["doc_id", "title"]])
display(questions_df[["question_id", "question", "expected_doc", "answerable"]])

## Coverage and answerability
The plots below describe the small, synthetic evaluation set. They are descriptive checks—not evidence that the system will perform similarly on production traffic.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
docs_df["title"].value_counts().sort_values().plot(kind="barh", ax=axes[0], title="Knowledge-base documents")
questions_df["answerable"].value_counts().rename(index={True: "Answerable", False: "Unanswerable"}).plot(
    kind="bar", ax=axes[1], title="Evaluation question answerability"
)
axes[0].set_xlabel("Document count")
axes[1].set_ylabel("Question count")
plt.tight_layout()
plt.show()

print("Question count by expected document:")
display(questions_df["expected_doc"].fillna("No expected document").value_counts().rename_axis("document").to_frame("questions"))

## Initial observations
- The corpus is deliberately small and covers only the topics represented in the sample documents.
- Evaluation questions are labelled with an expected source document, expected answer terms, and answerability.
- A useful next step is to test whether retrieval surfaces the expected source before evaluating generated answers.
- Real deployment would require a larger, reviewed corpus, representative support questions, privacy review, and human-labelled quality measures.